# Lab 10-05: Agent Queries

**Notebook 4 of 4.** Sends research queries to the `arxiv-nlp-agent` via the Responses API
and renders grounded, cited responses as structured citation cards.

The agent has two MCP tools and routes each query to the appropriate KB:

```
User query
  └─► arxiv-nlp-agent  (versioned, via Responses API)
        ├─► MCPTool kb_fast → arxiv-nlp-kb-fast  (simple / single-topic questions)
        │     └─► direct semantic search → arxiv-nlp index → citations
        └─► MCPTool kb_standard → arxiv-nlp-kb   (complex / multi-part questions)
              └─► query planning: gpt-4.1-mini via APIM
                    └─► parallel sub-queries → arxiv-nlp index
                          └─► semantic rerank → merged results → citations
  └─► Agent synthesises final response with citations
```

**Query types demonstrated:**
1. **Multi-intent** — complex question; expect `kb_standard` (LLM query decomposition)
2. **Temporal reasoning** — implicit year constraint; expect `kb_standard`
3. **Cross-lingual / low-resource** — vocabulary-rich; expect `kb_standard`
4. **Out-of-scope** — grounding enforcement; expect `"I don't know."`
5. **Simple factual** — single-topic lookup; expect `kb_fast` (direct retrieval)

## Prerequisites

1. **Run `10-03-knowledge-base-setup.ipynb`** — both KBs, MCP connections, and the agent must exist.
2. **`.env` file** — `IQ_AGENT_VERSION` is written automatically by `10-03-knowledge-base-setup.ipynb`
   (phase 5). `IQ_FOUNDRY_PROJECT_ENDPOINT` is written by `10-01-deploy-search-and-project.ipynb`.
   ```
   IQ_FOUNDRY_PROJECT_ENDPOINT=https://aif-spoke-multi-{suffix}.services.ai.azure.com/api/projects/iq-project
   IQ_AGENT_VERSION=<written automatically by 10-03 phase 5>
   ```
3. **Python environment** — `uv sync`, select the `.venv` kernel.
4. **Azure CLI** — `az login`.
5. **RBAC** — your identity needs **Azure AI Developer** on the Foundry project.

## 1. Imports and configuration

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from display_helpers import show_citation_cards, show_error

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

AGENT_NAME    = 'arxiv-nlp-agent'
AGENT_VERSION = os.environ.get('IQ_AGENT_VERSION', '1')

project_endpoint = os.environ['IQ_FOUNDRY_PROJECT_ENDPOINT']

print(f'Project endpoint: {project_endpoint}')
print(f'Agent name      : {AGENT_NAME}')
print(f'Agent version   : {AGENT_VERSION}')

Project endpoint: https://aif-spoke-multi-gvwiex.services.ai.azure.com/api/projects/iq-project
Agent name      : arxiv-nlp-agent
Agent version   : 2


## 2. Create clients

In [4]:
credential     = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client  = project_client.get_openai_client()

print('Project client: ready')
print('OpenAI client : ready')

Project client: ready
OpenAI client : ready


## 3. Query helper

`ask_agent` invokes the versioned agent via the Responses API and returns the response
text together with the list of MCP tool labels the agent called. Each label corresponds
to one of the two KB tools configured in `10-03-knowledge-base-setup.ipynb`:

| Label | KB | Effort |
|-------|----|--------|
| `kb_fast` | `arxiv-nlp-kb-fast` | `minimal` — direct retrieval, no LLM |
| `kb_standard` | `arxiv-nlp-kb` | `low` — LLM query planning via APIM |

The `server_label` comes from the `mcp_call` items in `response.output`, which record
every tool invocation made during the response. Observing which label appears tells you
whether the agent's routing decision matched the expected complexity of the query.

In [5]:
def ask_agent(question: str) -> tuple[str, list[str]]:
    """Send a question to the arxiv-nlp-agent.

    Returns:
        (response_text, tools_used) where tools_used is the list of MCP server_label
        values from mcp_call items in response.output — shows which KB the agent chose.
    """
    response = openai_client.responses.create(
        input=question,
        extra_body={
            'agent_reference': {
                'name': AGENT_NAME,
                'version': AGENT_VERSION,  # explicit version — pins to setup in 10-03
                'type': 'agent_reference',
            }
        },
    )
    tools_used = [
        item.server_label
        for item in response.output
        if getattr(item, 'type', None) == 'mcp_call' and getattr(item, 'server_label', None)
    ]
    return response.output_text, tools_used

---
## Query 1 — Multi-intent query (tests query decomposition)

A compound question spanning two distinct NLP topics. Expect the agent to call
**`kb_standard`** — the low-effort KB's LLM planning pass decomposes this into
focused sub-queries and fans them out in parallel.

This query tests:
- Sub-query decomposition across two distinct NLP topics
- Parallel retrieval fan-out and result merging
- Citation attribution across multiple papers

In [6]:
Q1 = (
    'How have attention mechanisms evolved from the original Bahdanau attention to modern '
    'transformers, and what are the key architectural innovations that enabled improvements '
    'in machine translation quality?'
)

text1, tools1 = ask_agent(Q1)
print(f'Tools called: {tools1}')  # expect: kb_standard
show_citation_cards(Q1, text1)

Tools called: ['kb_standard']


**Query:** *"How have attention mechanisms evolved from the original Bahdanau attention to modern transformers, and what are the key architectural innovations that enabled improvements in machine translation quality?"*

---
## Query 2 — Temporal reasoning (tests year-based retrieval)

A question with an implicit temporal constraint. Expect **`kb_standard`** — the LLM
planning pass can interpret "after 2020" and weight recency in sub-query construction.

This query tests:
- Temporal sensitivity of the retrieval pipeline
- Whether `low` reasoning effort correctly weights recency
- OData-filterable year field in semantic context

In [7]:
Q2 = (
    'What are the most recent advances in large language model pre-training techniques '
    'published after 2020, particularly around scaling laws and data efficiency?'
)

text2, tools2 = ask_agent(Q2)
print(f'Tools called: {tools2}')  # expect: kb_standard
show_citation_cards(Q2, text2)

Tools called: ['kb_standard']


**Query:** *"What are the most recent advances in large language model pre-training techniques published after 2020, particularly around scaling laws and data efficiency?"*

---
## Query 3 — Cross-lingual and low-resource NLP (tests semantic depth)

A semantically rich multi-part query where vocabulary varies substantially across papers
("low-resource", "data-scarce", "zero-shot cross-lingual"). Expect **`kb_standard`** —
BM25 degrades on vocabulary variation; the LLM planning pass surfaces the right papers
via semantic sub-query routing.

This query tests:
- Semantic retrieval across vocabulary variation
- Cross-lingual domain breadth in the corpus
- Citation quality and relevance of returned papers

In [8]:
Q3 = (
    'What techniques have been proposed for cross-lingual transfer learning in low-resource '
    'languages, and how do multilingual pretrained models compare to language-specific models '
    'for morphologically rich languages?'
)

text3, tools3 = ask_agent(Q3)
print(f'Tools called: {tools3}')  # expect: kb_standard
show_citation_cards(Q3, text3)

Tools called: ['kb_standard']


**Query:** *"What techniques have been proposed for cross-lingual transfer learning in low-resource languages, and how do multilingual pretrained models compare to language-specific models for morphologically rich languages?"*

---
## Query 4 — Out-of-scope question (tests grounding enforcement)

Verifies that the agent stays grounded — it should respond with `"I don't know."` for
questions not covered by the knowledge base. The agent may call either tool; neither
should return relevant papers, so the instructions enforce the fallback response.

In [ ]:
Q4 = 'What are the main ingredients in a traditional French cassoulet?'

text4, tools4 = ask_agent(Q4)
print(f'Tools called: {tools4}')
show_citation_cards(Q4, text4)

---
## Query 5 — Simple factual lookup (tests `kb_fast` routing)

A narrow, single-topic question with no decomposition needed. Expect the agent to call
**`kb_fast`** — there is no benefit to LLM query planning for a direct factual lookup,
and the instructions guide the agent to prefer the fast tool for these cases.

This query tests:
- Agent routing to `kb_fast` for simple questions
- `minimal` effort direct retrieval quality
- Contrast with Q1–Q3 where `kb_standard` was warranted

In [ ]:
Q5 = 'What did Vaswani et al. propose in the "Attention Is All You Need" paper?'

text5, tools5 = ask_agent(Q5)
print(f'Tools called: {tools5}')  # expect: kb_fast
show_citation_cards(Q5, text5)